# IDBD (autostep) on multi-MNIST 2-task with sparse LTU hidden layer

Sanity-check per-weight step-size meta-learning on the same architecture
used for the DEEP R progression. Masks `M1`/`M2` are frozen at their sparse
init — no pruning, no growth — so the only difference vs. the baseline run
is the optimizer. Loss is MSE on logits vs one-hot targets so that the
standard regression-IDBD prediction-gradient interpretation applies (the
per-weight prediction gradient for a linear layer is the input feature
feeding that weight). Optimizer is
`optax_idbd(autostep=True, version='prediction_grads')` from
`phd.jax_core.optimizers.idbd`.

We log the mean per-weight step-size α = exp(β), broken down by layer
(W1 / W2) × task relationship (within / cross), averaged over **active**
connections in each category.

## Setup

In [1]:
import os
import sys

REPO_ROOT = '/home/edan/local_projects/phd_research'
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'phd', 'structure_search')):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from phd.jax_core.models import ltu

# Multi-MNIST 2-task layout
N_TASKS = 2
NUM_CLASSES = 10
INPUT_PER_TASK = 784
INPUT_DIM = INPUT_PER_TASK * N_TASKS              # 1568
OUTPUT_DIM = NUM_CLASSES * N_TASKS                # 20

# 2-layer LTU layout — each hidden unit hardwired to one output, equally split.
N_HIDDEN = 60                                     # must divide OUTPUT_DIM
HIDDEN_PER_OUTPUT = N_HIDDEN // OUTPUT_DIM        # 5
INPUT_FANIN = 128                                 # initial active input connections per hidden unit

assert N_HIDDEN % OUTPUT_DIM == 0, "N_HIDDEN must be divisible by OUTPUT_DIM"

print('JAX device:', jax.devices()[0])
print(f'INPUT_DIM={INPUT_DIM}  N_HIDDEN={N_HIDDEN}  OUTPUT_DIM={OUTPUT_DIM}'
      f'  HIDDEN_PER_OUTPUT={HIDDEN_PER_OUTPUT}  INPUT_FANIN={INPUT_FANIN}')

JAX device: cuda:0
INPUT_DIM=1568  N_HIDDEN=60  OUTPUT_DIM=20  HIDDEN_PER_OUTPUT=3  INPUT_FANIN=128


## Data

In [2]:
def load_data():
    """MNIST standardized per-pixel."""
    from data import load_dataset
    images, labels, _, _ = load_dataset('mnist', split='train')
    images = np.asarray(images, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int32)
    mean = images.mean(axis=0, keepdims=True)
    std = images.std(axis=0, keepdims=True)
    normalized = (images - mean) / np.maximum(std, 1e-3)
    return jnp.asarray(normalized), jnp.asarray(labels)


images, labels = load_data()
print('images:', images.shape, '   labels:', labels.shape)

images: (60000, 784)    labels: (60000,)


## Architecture

Forward: `z1 = x @ (W1 * M1); h = ltu(z1); logits = h @ (W2 * M2)`. Loss is
MSE on logits vs one-hot targets, averaged over both tasks. No biases.

In [3]:
def forward(W1, M1, W2, M2, x):
    z1 = x @ (W1 * M1)                              # (N_HIDDEN,)
    h = jax.nn.leaky_relu(z1)                                     # binary {0,1} forward, sigmoid-STE backward
    logits = h @ (W2 * M2)                          # (OUTPUT_DIM,)
    return logits, h, z1


def loss_fn(W1, M1, W2, M2, x, y):
    """MSE on logits vs one-hot targets, averaged across both tasks."""
    logits, _, _ = forward(W1, M1, W2, M2, x)
    logits_pt = logits.reshape(N_TASKS, NUM_CLASSES)
    targets = jax.nn.one_hot(y, NUM_CLASSES)
    return 0.5 * jnp.mean(jnp.square(logits_pt - targets))


def make_sample(images, labels, key):
    k1, k2 = jax.random.split(key)
    idx1 = jax.random.randint(k1, (), 0, images.shape[0])
    idx2 = jax.random.randint(k2, (), 0, images.shape[0])
    x = jnp.concatenate([images[idx1], images[idx2]])
    y = jnp.array([labels[idx1], labels[idx2]])
    return x, y

## Init

Per hidden unit: pick `INPUT_FANIN` random input pixels; one-hot to its assigned
output. Weights at active entries are Kaiming-uniform with the appropriate fan-in.
Future extensions (more outgoing connections, feature growth) just modify these
masks before passing them to the train functions.

In [4]:
def init_2layer_ltu(seed=0, n_hidden=N_HIDDEN, input_fanin=INPUT_FANIN):
    """Init W1, M1, W2, M2.

    M1: each column has `input_fanin` ones at random rows.
    M2: one-hot row, hidden unit i routes to output i // (n_hidden // OUTPUT_DIM).
    """
    k = jax.random.key(seed)
    k_m1, k_w1, k_w2 = jax.random.split(k, 3)

    # M1: per hidden unit, sample input_fanin random inputs.
    keys = jax.random.split(k_m1, n_hidden)

    def per_unit(key):
        noise = jax.random.uniform(key, (INPUT_DIM,))
        idx = jnp.argsort(-noise)[:input_fanin]
        return jnp.zeros(INPUT_DIM, dtype=jnp.int32).at[idx].set(1)

    M1_T = jax.vmap(per_unit)(keys)                              # (N_HIDDEN, INPUT_DIM)
    M1 = M1_T.T                                                  # (INPUT_DIM, N_HIDDEN)

    w1_bound = jnp.sqrt(3.0 / float(input_fanin))
    W1 = jax.random.uniform(k_w1, (INPUT_DIM, n_hidden),
                            minval=-w1_bound, maxval=w1_bound) * M1

    hidden_per_output = n_hidden // OUTPUT_DIM
    h2o = jnp.arange(n_hidden) // hidden_per_output              # (N_HIDDEN,)
    M2 = jax.nn.one_hot(h2o, OUTPUT_DIM, dtype=jnp.int32)        # (N_HIDDEN, OUTPUT_DIM)
    w2_bound = jnp.sqrt(3.0 / float(hidden_per_output))
    W2 = jax.random.uniform(k_w2, (n_hidden, OUTPUT_DIM),
                            minval=-w2_bound, maxval=w2_bound) * M2
    return W1, M1, W2, M2


# Sanity-check the init.
_W1, _M1, _W2, _M2 = init_2layer_ltu(seed=0)
print(f'M1 active per hidden unit: min={int(_M1.sum(0).min())} '
      f'mean={float(_M1.sum(0).mean())} max={int(_M1.sum(0).max())}')
print(f'M2 active per hidden unit: {int(_M2.sum(1).min())} (should be 1)')
print(f'M2 active per output: {int(_M2.sum(0).min())}/{int(_M2.sum(0).max())} '
      f'(should both be {HIDDEN_PER_OUTPUT})')

M1 active per hidden unit: min=128 mean=128.0 max=128
M2 active per hidden unit: 1 (should be 1)
M2 active per output: 3/3 (should both be 3)


## Training with IDBD (autostep, prediction grads)

Sparse-network forward, frozen masks. Loss is MSE on logits.

Each step:

1. compute loss gradients g1, g2 (only active entries are non-zero, since
   forward uses `W * M`); also evaluate `h = ltu(z1)` for use as a
   prediction gradient
2. mask the gradients with M1/M2 (defensive — inactive entries should not
   accumulate optimizer state)
3. form per-layer prediction gradients — for a linear layer the
   prediction gradient of weight `W[i,k]` is the input feature feeding
   it: `x_i` for W1, `h_k` for W2. Broadcast to fill the parameter shape
   and mask with M1/M2
4. transpose weights, grads, and prediction grads to `(n_out, n_in)` so
   autostep's `sum(α · pred_g², axis=-1)` runs over the fan-in
   (per-output-row normalization)
5. step IDBD; transpose updates back; re-apply M1/M2

`version='prediction_grads'` uses `pred_g²` as the curvature term — for
each linear layer this is exactly the per-weight Gauss-Newton diagonal
under squared-error loss. See
`phd.jax_core.optimizers.idbd.optax_idbd`.

In [5]:
from phd.jax_core.optimizers import optax_idbd


# Within/cross-task masks for W1 (input × hidden) and W2 (hidden × output).
INPUT_TASK_J  = jnp.arange(INPUT_DIM) // INPUT_PER_TASK
HIDDEN_TASK_J = (jnp.arange(N_HIDDEN) // HIDDEN_PER_OUTPUT) // NUM_CLASSES
OUTPUT_TASK_J = jnp.arange(OUTPUT_DIM) // NUM_CLASSES
SAME_TASK_IH_J = (INPUT_TASK_J[:, None]  == HIDDEN_TASK_J[None, :])    # (IN, HIDDEN)
SAME_TASK_HO_J = (HIDDEN_TASK_J[:, None] == OUTPUT_TASK_J[None, :])    # (HIDDEN, OUT)


def train_idbd_autostep(W1_init, M1_init, W2_init, M2_init, images, labels, *,
                        init_lr=1e-5,
                        meta_lr=5e-3,
                        tau=1e4,
                        weight_decay=0.0,
                        n_steps=500_000,
                        snapshot_every=2_000,
                        permute_period=0,
                        seed=0):
    """IDBD (autostep, version='prediction_grads') on the sparse LTU network
    with frozen masks. MSE loss on logits.

    Per-layer prediction gradients are the input features feeding each
    weight (`x` for W1, `h` for W2), broadcast and masked. Weights are
    handed to the optimizer in `(n_out, n_in)` form so autostep's
    `sum(α · pred_g², axis=-1)` runs over the fan-in axis.

    Snapshots per chunk: avg loss, mean α, mean β, and mean |W| over active
    connections in each (layer × within/cross) category. Also saves the
    final weights and the final β tensors for end-of-training distribution
    plots.
    """
    n_chunks = n_steps // snapshot_every
    perm0_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    perm1_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)

    M1 = M1_init.astype(jnp.float32)
    M2 = M2_init.astype(jnp.float32)

    optimizer = optax_idbd(
        meta_lr=meta_lr, init_lr=init_lr,
        weight_decay=weight_decay,
        autostep=True, tau=tau,
        version='prediction_grads',
    )
    init_opt_state = optimizer.init((W1_init.T, W2_init.T))

    init_carry = (W1_init, W2_init, init_opt_state,
                  perm0_init, perm1_init,
                  jnp.array(0, dtype=jnp.int32))

    cat_w1_within = M1 * SAME_TASK_IH_J.astype(jnp.float32)
    cat_w1_cross  = M1 * (~SAME_TASK_IH_J).astype(jnp.float32)
    cat_w2_within = M2 * SAME_TASK_HO_J.astype(jnp.float32)
    cat_w2_cross  = M2 * (~SAME_TASK_HO_J).astype(jnp.float32)

    def step_fn(carry, key):
        W1, W2, opt_state, perm0, perm1, t = carry
        data_key, perm_key = jax.random.split(key)
        x, y_raw = make_sample(images, labels, data_key)
        y = jnp.array([perm0[y_raw[0]], perm1[y_raw[1]]])

        def _loss_and_h(W1_, W2_):
            logits, h_, _ = forward(W1_, M1, W2_, M2, x)
            logits_pt = logits.reshape(N_TASKS, NUM_CLASSES)
            targets = jax.nn.one_hot(y, NUM_CLASSES)
            loss_ = 0.5 * jnp.mean(jnp.square(logits_pt - targets))
            return loss_, h_

        (loss, h), (g1, g2) = jax.value_and_grad(
            _loss_and_h, argnums=(0, 1), has_aux=True,
        )(W1, W2)

        # Mask grads -- inactive entries should never accumulate optimizer state.
        g1 = g1 * M1
        g2 = g2 * M2

        # Prediction grads: per-layer input features broadcast over the
        # output axis, then masked. Treats each linear layer as an
        # independent linear regression for step-size accounting.
        pred_g1 = M1 * x[:, None]   # (INPUT, HIDDEN)  -- x broadcast along HIDDEN
        pred_g2 = M2 * h[:, None]   # (HIDDEN, OUTPUT) -- h broadcast along OUTPUT

        # Transpose to (n_out, n_in) for autostep's fan-in normalization.
        grads_T      = (g1.T,      g2.T)
        pred_grads_T = (pred_g1.T, pred_g2.T)
        updates_T, opt_state = optimizer.update(
            (grads_T, pred_grads_T), opt_state, (W1.T, W2.T),
        )
        W1 = (W1 + updates_T[0].T) * M1
        W2 = (W2 + updates_T[1].T) * M2

        t_next = t + 1
        if permute_period > 0:
            should_perm = (t_next >= permute_period) & (t_next % permute_period == 0)
            pk1, pk2 = jax.random.split(perm_key)
            which = jax.random.randint(pk1, (), 0, N_TASKS)
            new_perm = jax.random.permutation(pk2, NUM_CLASSES).astype(jnp.int32)
            perm0 = jnp.where(should_perm & (which == 0), new_perm, perm0)
            perm1 = jnp.where(should_perm & (which == 1), new_perm, perm1)

        return (W1, W2, opt_state, perm0, perm1, t_next), loss

    def _masked_mean(arr, mask):
        return jnp.sum(arr * mask) / jnp.maximum(jnp.sum(mask), 1.0)

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, losses = jax.lax.scan(step_fn, carry, keys)
        W1_now, W2_now, opt_state, _p0, _p1, t = carry
        beta_W1_T, beta_W2_T = opt_state.beta
        # Transpose back to original (n_in, n_out) layout for category masks.
        beta1  = beta_W1_T.T
        beta2  = beta_W2_T.T
        alpha1 = jnp.exp(beta1)
        alpha2 = jnp.exp(beta2)
        abs_W1 = jnp.abs(W1_now)
        abs_W2 = jnp.abs(W2_now)
        snap = dict(
            step=t,
            avg_loss=losses.mean(),
            mean_alpha_W1_within=_masked_mean(alpha1, cat_w1_within),
            mean_alpha_W1_cross =_masked_mean(alpha1, cat_w1_cross),
            mean_alpha_W2_within=_masked_mean(alpha2, cat_w2_within),
            mean_alpha_W2_cross =_masked_mean(alpha2, cat_w2_cross),
            mean_beta_W1_within =_masked_mean(beta1, cat_w1_within),
            mean_beta_W1_cross  =_masked_mean(beta1, cat_w1_cross),
            mean_beta_W2_within =_masked_mean(beta2, cat_w2_within),
            mean_beta_W2_cross  =_masked_mean(beta2, cat_w2_cross),
            mean_wL1_W1_within  =_masked_mean(abs_W1, cat_w1_within),
            mean_wL1_W1_cross   =_masked_mean(abs_W1, cat_w1_cross),
            mean_wL1_W2_within  =_masked_mean(abs_W2, cat_w2_within),
            mean_wL1_W2_cross   =_masked_mean(abs_W2, cat_w2_cross),
        )
        return carry, snap

    rng = jax.random.key(seed)
    chunk_keys = jax.random.split(rng, n_chunks)
    final_carry, snaps = jax.lax.scan(chunk_fn, init_carry, chunk_keys)
    snaps = {k: jax.device_get(v) for k, v in snaps.items()}
    snaps['final_W1'] = jax.device_get(final_carry[0])
    snaps['final_W2'] = jax.device_get(final_carry[1])
    final_opt_state = final_carry[2]
    snaps['final_beta_W1'] = jax.device_get(final_opt_state.beta[0].T)  # (INPUT, HIDDEN)
    snaps['final_beta_W2'] = jax.device_get(final_opt_state.beta[1].T)  # (HIDDEN, OUTPUT)
    return snaps

## Baseline (plain SGD, frozen masks)

Same init, plain SGD with frozen masks — the no-MetaOptimize reference loss
curve.

In [6]:
def train_baseline(W1_init, M1_init, W2_init, M2_init, images, labels, *,
                   lr=2**-7,
                   n_steps=500_000,
                   snapshot_every=2_000,
                   permute_period=0,
                   seed=0):
    """Plain SGD with frozen masks. No L1, no noise, no deactivation.

    `permute_period`: 0 = stationary. >0 = same non-stationary mechanism as
    train_deep_r so the loss curves are directly comparable."""
    n_chunks = n_steps // snapshot_every
    perm0_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    perm1_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    init_carry = (W1_init, W2_init, perm0_init, perm1_init,
                  jnp.array(0, dtype=jnp.int32))

    def step_fn(carry, key):
        W1, W2, perm0, perm1, t = carry
        data_key, perm_key = jax.random.split(key)
        x, y_raw = make_sample(images, labels, data_key)
        y = jnp.array([perm0[y_raw[0]], perm1[y_raw[1]]])
        def _loss(W1_, W2_):
            return loss_fn(W1_, M1_init, W2_, M2_init, x, y)
        loss, (g1, g2) = jax.value_and_grad(_loss, argnums=(0, 1))(W1, W2)
        W1 = (W1 - lr * g1) * M1_init
        W2 = (W2 - lr * g2) * M2_init
        t_next = t + 1
        if permute_period > 0:
            should_perm = (t_next >= permute_period) & (t_next % permute_period == 0)
            pk1, pk2 = jax.random.split(perm_key)
            which = jax.random.randint(pk1, (), 0, N_TASKS)
            new_perm = jax.random.permutation(pk2, NUM_CLASSES).astype(jnp.int32)
            perm0 = jnp.where(should_perm & (which == 0), new_perm, perm0)
            perm1 = jnp.where(should_perm & (which == 1), new_perm, perm1)
        return (W1, W2, perm0, perm1, t_next), loss

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, losses = jax.lax.scan(step_fn, carry, keys)
        W1, W2, _p0, _p1, t = carry
        return carry, dict(step=t, avg_loss=losses.mean())

    rng = jax.random.key(seed)
    chunk_keys = jax.random.split(rng, n_chunks)
    final_carry, snaps = jax.lax.scan(chunk_fn, init_carry, chunk_keys)
    snaps = {k: jax.device_get(v) for k, v in snaps.items()}
    snaps['final_W1'] = jax.device_get(final_carry[0])
    snaps['final_W2'] = jax.device_get(final_carry[1])
    return snaps

## Run

The IDBD run starts with a deliberately low initial step-size
(`init_lr=1e-5`); autostep then ramps the per-weight step-sizes up
subject to its `Σ α · pred_g² ≤ 1` clip per output neuron. Meta step-size
is `meta_lr=5e-3`.

In [7]:
# Shared init (same seed -> same starting topology and weights for both runs).
W1_init, M1_init, W2_init, M2_init = init_2layer_ltu(seed=0)

IDBD_CONFIG = dict(
    init_lr=2**-6,#1e-4,
    meta_lr=0.005,
    tau=1e4,
    weight_decay=0.0,
    n_steps=500_000,
    snapshot_every=2_000,
    permute_period=0,
    seed=0,
)
BASELINE_LR = 2**-6

train_idbd_jit = jax.jit(
    train_idbd_autostep,
    static_argnames=('init_lr', 'meta_lr', 'tau', 'weight_decay',
                     'n_steps', 'snapshot_every', 'permute_period', 'seed'),
)
train_baseline_jit = jax.jit(
    train_baseline,
    static_argnames=('lr', 'n_steps', 'snapshot_every', 'permute_period', 'seed'),
)

print('Running IDBD (autostep)...')
meta_snaps = train_idbd_jit(W1_init, M1_init, W2_init, M2_init,
                             images, labels, **IDBD_CONFIG)
print(f'  final loss: {float(meta_snaps["avg_loss"][-1]):.4f}')
print(f'  final mean α  W1 within / cross: '
      f'{float(meta_snaps["mean_alpha_W1_within"][-1]):.3e} '
      f'/ {float(meta_snaps["mean_alpha_W1_cross"][-1]):.3e}')
print(f'  final mean α  W2 within / cross: '
      f'{float(meta_snaps["mean_alpha_W2_within"][-1]):.3e} '
      f'/ {float(meta_snaps["mean_alpha_W2_cross"][-1]):.3e}')

print('\nRunning baseline (plain SGD, frozen masks)...')
baseline_snaps = train_baseline_jit(W1_init, M1_init, W2_init, M2_init, images, labels,
                                     lr=BASELINE_LR,
                                     n_steps=IDBD_CONFIG['n_steps'],
                                     snapshot_every=IDBD_CONFIG['snapshot_every'],
                                     permute_period=IDBD_CONFIG['permute_period'],
                                     seed=IDBD_CONFIG['seed'])
print(f'  final loss: {float(baseline_snaps["avg_loss"][-1]):.4f}')

Found multiple sets of weights, but AutoStep does not support non-linear  layer structures. If the weights provided to AutoStep are stacked and not independent, then this will probably cause a silent bug.


Running IDBD (autostep)...
  final loss: 0.0415
  final mean α  W1 within / cross: 5.976e-04 / 2.275e-05
  final mean α  W2 within / cross: 9.986e-04 / 0.000e+00

Running baseline (plain SGD, frozen masks)...
  final loss: 0.0205


## Step-size traces

The four mean step-sizes are sampled at chunk boundaries. Within-task
entries should ramp up faster than cross-task entries when the latter
provide little signal for the per-task softmax loss.

Note: with the current architecture each hidden unit is hardwired to a
single within-task output, so `M2` has no cross-task active entries and
the W2-cross trace is identically zero by construction.

In [8]:
def alpha_traces(snaps):
    return dict(
        steps=np.asarray(snaps['step']),
        W1_within=np.asarray(snaps['mean_alpha_W1_within']),
        W1_cross =np.asarray(snaps['mean_alpha_W1_cross']),
        W2_within=np.asarray(snaps['mean_alpha_W2_within']),
        W2_cross =np.asarray(snaps['mean_alpha_W2_cross']),
    )


def beta_traces(snaps):
    return dict(
        steps=np.asarray(snaps['step']),
        W1_within=np.asarray(snaps['mean_beta_W1_within']),
        W1_cross =np.asarray(snaps['mean_beta_W1_cross']),
        W2_within=np.asarray(snaps['mean_beta_W2_within']),
        W2_cross =np.asarray(snaps['mean_beta_W2_cross']),
    )


def w_l1_traces(snaps):
    return dict(
        steps=np.asarray(snaps['step']),
        W1_within=np.asarray(snaps['mean_wL1_W1_within']),
        W1_cross =np.asarray(snaps['mean_wL1_W1_cross']),
        W2_within=np.asarray(snaps['mean_wL1_W2_within']),
        W2_cross =np.asarray(snaps['mean_wL1_W2_cross']),
    )


traces = alpha_traces(meta_snaps)
betas  = beta_traces(meta_snaps)
wl1s   = w_l1_traces(meta_snaps)

print(f'final mean α   W1 within / cross: '
      f'{traces["W1_within"][-1]:.3e} / {traces["W1_cross"][-1]:.3e}')
print(f'final mean α   W2 within / cross: '
      f'{traces["W2_within"][-1]:.3e} / {traces["W2_cross"][-1]:.3e}')
print(f'final mean β   W1 within / cross: '
      f'{betas["W1_within"][-1]:.3f} / {betas["W1_cross"][-1]:.3f}')
print(f'final mean β   W2 within / cross: '
      f'{betas["W2_within"][-1]:.3f} / {betas["W2_cross"][-1]:.3f}')
print(f'final mean |W| W1 within / cross: '
      f'{wl1s["W1_within"][-1]:.3e} / {wl1s["W1_cross"][-1]:.3e}')
print(f'final mean |W| W2 within / cross: '
      f'{wl1s["W2_within"][-1]:.3e} / {wl1s["W2_cross"][-1]:.3e}')

final mean α   W1 within / cross: 5.976e-04 / 2.275e-05
final mean α   W2 within / cross: 9.986e-04 / 0.000e+00
final mean β   W1 within / cross: -11.610 / -12.115
final mean β   W2 within / cross: -6.992 / 0.000
final mean |W| W1 within / cross: 8.453e-02 / 7.661e-02
final mean |W| W2 within / cross: 4.720e-02 / 0.000e+00


## Plots

In [9]:
WITHIN_COLOR = '#1f77b4'   # blue
CROSS_COLOR  = '#d62728'   # red


def plot_loss(meta_snaps, baseline_snaps):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.asarray(meta_snaps['step']),
                             y=np.asarray(meta_snaps['avg_loss']),
                             mode='lines', name='IDBD (autostep)'))
    fig.add_trace(go.Scatter(x=np.asarray(baseline_snaps['step']),
                             y=np.asarray(baseline_snaps['avg_loss']),
                             mode='lines', name='baseline SGD',
                             line=dict(dash='dot')))
    fig.update_layout(title='Loss over training',
                      xaxis_title='step',
                      yaxis_title='mean loss over snapshot',
                      width=900, height=420)
    fig.show()
    return fig


def _plot_2panel_traces(traces, *, title, ylabel, shared_yaxes=True, log_y=False):
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=('W1 (incoming)', 'W2 (outgoing)'),
                        shared_yaxes=shared_yaxes)
    s = traces['steps']
    fig.add_trace(go.Scatter(x=s, y=traces['W1_within'], mode='lines',
                             name='W1 within',
                             line=dict(color=WITHIN_COLOR)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=s, y=traces['W1_cross'],  mode='lines',
                             name='W1 cross',
                             line=dict(color=CROSS_COLOR)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=s, y=traces['W2_within'], mode='lines',
                             name='W2 within',
                             line=dict(color=WITHIN_COLOR, dash='dash')),
                  row=1, col=2)
    fig.add_trace(go.Scatter(x=s, y=traces['W2_cross'],  mode='lines',
                             name='W2 cross',
                             line=dict(color=CROSS_COLOR, dash='dash')),
                  row=1, col=2)
    if log_y:
        fig.update_yaxes(type='log')
    fig.update_xaxes(title_text='step', row=1, col=1)
    fig.update_xaxes(title_text='step', row=1, col=2)
    fig.update_yaxes(title_text=ylabel, row=1, col=1)
    fig.update_layout(title=title, width=1000, height=420)
    fig.show()
    return fig


def plot_alpha_traces(traces):
    """Mean per-weight step-size α = exp(β), split by layer × task relation."""
    return _plot_2panel_traces(
        traces,
        title='Mean step-size α = exp(β) over active connections',
        ylabel='mean α',
        shared_yaxes=True, log_y=True,
    )


def plot_beta_traces(traces):
    """Mean per-weight log step-size β = log α, split by layer × task relation."""
    return _plot_2panel_traces(
        traces,
        title='Mean log step-size β = log α over active connections',
        ylabel='mean β',
        shared_yaxes=True, log_y=False,
    )


def plot_w_l1_traces(traces):
    """Mean per-weight |W|, split by layer × task relation."""
    return _plot_2panel_traces(
        traces,
        title='Mean |W| over active connections',
        ylabel='mean |W|',
        shared_yaxes=False, log_y=False,
    )


def _category_indices(M1, M2):
    M1_b = np.asarray(M1).astype(bool)
    M2_b = np.asarray(M2).astype(bool)
    same_ih = np.asarray(SAME_TASK_IH_J)
    same_ho = np.asarray(SAME_TASK_HO_J)
    return dict(
        W1_within=M1_b & same_ih,
        W1_cross =M1_b & ~same_ih,
        W2_within=M2_b & same_ho,
        W2_cross =M2_b & ~same_ho,
    )


def _add_hist_pair(fig, vals_within, vals_cross, *, row, col, nbins, name_prefix):
    """Overlaid within/cross histograms with shared bin edges per panel."""
    vals_within = np.asarray(vals_within).ravel()
    vals_cross  = np.asarray(vals_cross).ravel()
    combined = np.concatenate([vals_within, vals_cross])
    if combined.size == 0:
        return
    lo = float(np.min(combined))
    hi = float(np.max(combined))
    if hi <= lo:
        hi = lo + 1.0
    size = (hi - lo) / nbins
    # Nudge `end` past `hi` so the max sample isn't dropped from rounding.
    xbins = dict(start=lo, end=hi + size * 0.5, size=size)
    if vals_within.size > 0:
        fig.add_trace(go.Histogram(x=vals_within,
                                    name=f'{name_prefix} within',
                                    marker_color=WITHIN_COLOR, opacity=0.6,
                                    xbins=xbins, autobinx=False),
                      row=row, col=col)
    if vals_cross.size > 0:
        fig.add_trace(go.Histogram(x=vals_cross,
                                    name=f'{name_prefix} cross',
                                    marker_color=CROSS_COLOR, opacity=0.6,
                                    xbins=xbins, autobinx=False),
                      row=row, col=col)


def plot_alpha_distribution(meta_snaps, M1, M2, nbins=60):
    """Final-step distribution of α = exp(β), shown as log10(α)."""
    cats = _category_indices(M1, M2)
    final_alpha_W1 = np.exp(np.asarray(meta_snaps['final_beta_W1']))
    final_alpha_W2 = np.exp(np.asarray(meta_snaps['final_beta_W2']))
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=('W1 (incoming)', 'W2 (outgoing)'))
    _add_hist_pair(
        fig,
        np.log10(final_alpha_W1[cats['W1_within']]) if cats['W1_within'].any() else np.array([]),
        np.log10(final_alpha_W1[cats['W1_cross']])  if cats['W1_cross'].any()  else np.array([]),
        row=1, col=1, nbins=nbins, name_prefix='W1',
    )
    _add_hist_pair(
        fig,
        np.log10(final_alpha_W2[cats['W2_within']]) if cats['W2_within'].any() else np.array([]),
        np.log10(final_alpha_W2[cats['W2_cross']])  if cats['W2_cross'].any()  else np.array([]),
        row=1, col=2, nbins=nbins, name_prefix='W2',
    )
    fig.update_xaxes(title_text='log₁₀ α', row=1, col=1)
    fig.update_xaxes(title_text='log₁₀ α', row=1, col=2)
    fig.update_yaxes(title_text='count', row=1, col=1)
    fig.update_layout(title='Final α distribution (log scale)',
                      barmode='overlay',
                      width=1000, height=420)
    fig.show()
    return fig


def plot_w_l1_distribution(meta_snaps, M1, M2, nbins=60):
    """Final-step distribution of |W| for each category."""
    cats = _category_indices(M1, M2)
    final_W1 = np.abs(np.asarray(meta_snaps['final_W1']))
    final_W2 = np.abs(np.asarray(meta_snaps['final_W2']))
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=('W1 (incoming)', 'W2 (outgoing)'))
    _add_hist_pair(
        fig,
        final_W1[cats['W1_within']] if cats['W1_within'].any() else np.array([]),
        final_W1[cats['W1_cross']]  if cats['W1_cross'].any()  else np.array([]),
        row=1, col=1, nbins=nbins, name_prefix='W1',
    )
    _add_hist_pair(
        fig,
        final_W2[cats['W2_within']] if cats['W2_within'].any() else np.array([]),
        final_W2[cats['W2_cross']]  if cats['W2_cross'].any()  else np.array([]),
        row=1, col=2, nbins=nbins, name_prefix='W2',
    )
    fig.update_xaxes(title_text='|W|', row=1, col=1)
    fig.update_xaxes(title_text='|W|', row=1, col=2)
    fig.update_yaxes(title_text='count', row=1, col=1)
    fig.update_layout(title='Final |W| distribution',
                      barmode='overlay',
                      width=1000, height=420)
    fig.show()
    return fig

In [10]:
plot_loss(meta_snaps, baseline_snaps)
plot_beta_traces(betas)
plot_w_l1_traces(wl1s)
plot_alpha_distribution(meta_snaps, M1_init, M2_init)
plot_w_l1_distribution(meta_snaps, M1_init, M2_init);